# ClinicalTrialEnv Quick Judge Demo

RL environment for clinical trial workflow execution. This notebook boots the environment, shows a short Task 3 interaction, loads precomputed evaluation artifacts, and visualizes baseline vs policy behavior without running heavy training.

- HF Space: [abisheiks/clinicaltrial-env](https://huggingface.co/spaces/abisheiks/clinicaltrial-env)
- GitHub: [abisheik687/clinicaltrial-env](https://github.com/abisheik687/clinicaltrial-env)
- Blog: [Why LLMs Fail at Clinical Workflows (and What RL Reveals)](https://github.com/abisheik687/clinicaltrial-env/blob/master/docs/blog.md)


In [ ]:
!git clone https://github.com/abisheik687/clinicaltrial-env.git || true
%cd /content/clinicaltrial-env
!pip install -q fastapi "uvicorn[standard]" "pydantic[email]>=2.11.7,<3" pydantic-settings faker numpy pyyaml python-multipart httpx openai matplotlib


In [ ]:
%cd /content/clinicaltrial-env
!python -m uvicorn server.main:app --host 0.0.0.0 --port 7860 > /content/clinicaltrialenv_server.log 2>&1 &

import time, httpx

health = None
for _ in range(30):
    try:
        resp = httpx.get('http://localhost:7860/health', timeout=5.0)
        if resp.status_code == 200:
            health = resp.json()
            break
    except Exception:
        pass
    time.sleep(1)

assert health is not None, 'Server did not become healthy in time'
print(health)
!tail -n 10 /content/clinicaltrialenv_server.log


In [ ]:
!curl -s http://localhost:7860/health


## Quick Environment Demo

The next cell resets Task 3 and executes five manual actions so judges can see state transitions, rewards, and workflow progression without training anything.


In [ ]:
import json, httpx

client = httpx.Client(base_url='http://localhost:7860', timeout=20.0)
reset = client.post('/reset', json={'task_id': 'task3', 'seed': 44}).json()
session_id = reset['session_id']
obs = reset['observation']
print('Initial phase:', obs['operational_state']['workflow_phase'])

manual_actions = [
    {
        'action_type': 'evaluate_criterion',
        'criterion_id': 'INC-001',
        'evaluation': {'criterion_id': 'INC-001', 'verdict': 'met', 'reasoning': 'Age is within range.'},
        'confidence_score': 0.9,
    },
    {
        'action_type': 'evaluate_criterion',
        'criterion_id': 'INC-002',
        'evaluation': {'criterion_id': 'INC-002', 'verdict': 'met', 'reasoning': 'MECP2 requirement satisfied.'},
        'confidence_score': 0.9,
    },
    {
        'action_type': 'ask_clarification',
        'clarification_target': 'INC-003',
        'confidence_score': 0.8,
    },
    {
        'action_type': 'evaluate_criterion',
        'criterion_id': 'INC-003',
        'evaluation': {'criterion_id': 'INC-003', 'verdict': 'met', 'reasoning': 'Clarified score remains in range.'},
        'confidence_score': 0.9,
    },
    {
        'action_type': 'evaluate_criterion',
        'criterion_id': 'INC-004',
        'evaluation': {'criterion_id': 'INC-004', 'verdict': 'met', 'reasoning': 'No conflicting prior therapy evidence visible.'},
        'confidence_score': 0.9,
    },
]

demo_log = []
for idx, action in enumerate(manual_actions, start=1):
    step = client.post('/step', json={'session_id': session_id, 'action': action}).json()
    demo_log.append({
        'step': idx,
        'action_type': action['action_type'],
        'criterion_id': action.get('criterion_id'),
        'reward': step['reward']['total_reward'],
        'done': step['done'],
        'phase': step['observation']['operational_state']['workflow_phase'],
        'amendment_active': step['observation']['trial_protocol_summary']['amendment_active'],
    })

print(json.dumps(demo_log, indent=2))


In [ ]:
%cd /content/clinicaltrial-env
!python training/evaluate_models.py --policy fallback --task-ids task3 --num-seeds 2 --output artifacts/eval/colab_quick_eval.json


## Load Precomputed Results

The next cell loads saved artifact files from the repository so judges can inspect baseline behavior, compact-policy behavior, and training evidence without waiting for a full run.


In [ ]:
from pathlib import Path
import json

base_eval = json.loads(Path('artifacts/eval/base_model_task3_eval.json').read_text())
compact_eval = json.loads(Path('artifacts/eval/policy_gradient_task3_eval.json').read_text())
training_log = json.loads(Path('artifacts/stepwise_llm_rl_tinydebug_run30_v2/train_history.json').read_text())
training_summary = json.loads(Path('artifacts/stepwise_llm_rl_tinydebug_run30_v2/summary.json').read_text())

loaded_summary = {
    'baseline_aggregate': base_eval['aggregate'],
    'compact_policy_aggregate': compact_eval['aggregate'],
    'training_summary': training_summary,
}
print(json.dumps(loaded_summary, indent=2))


## Visual Proof

These charts answer the key judge questions quickly: what failed, what improved, and whether any real RL signal exists.


In [ ]:
import matplotlib.pyplot as plt

labels = ['Baseline', 'Compact Policy']
rewards = [base_eval['aggregate']['mean_final_reward'], compact_eval['aggregate']['mean_final_reward']]
colors = ['#d9534f', '#2e8b57']

plt.figure(figsize=(6, 4))
bars = plt.bar(labels, rewards, color=colors)
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Baseline vs Trained Reward')
plt.ylabel('Mean Final Reward')
for bar, value in zip(bars, rewards):
    plt.text(bar.get_x() + bar.get_width()/2, value + (0.03 if value >= 0 else -0.08), f'{value:.2f}', ha='center', va='bottom' if value >= 0 else 'top')
plt.show()


In [ ]:
episodes = [row['episode'] for row in training_log]
reward_series = [row['reward'] for row in training_log]

plt.figure(figsize=(8, 4))
plt.plot(episodes, reward_series, marker='o', linewidth=1.8, color='#1e6bb8')
plt.title('Training Reward Curve')
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.grid(alpha=0.25)
plt.show()


In [ ]:
from IPython.display import Markdown, display

metrics_md = f'''
| System | Success Rate | Unsafe Rate | Mean Reward |
| --- | ---: | ---: | ---: |
| Baseline | {base_eval['aggregate']['success_rate']:.2f} | {base_eval['aggregate']['unsafe_rate']:.2f} | {base_eval['aggregate']['mean_final_reward']:.2f} |
| Compact Policy | {compact_eval['aggregate']['success_rate']:.2f} | {compact_eval['aggregate']['unsafe_rate']:.2f} | {compact_eval['aggregate']['mean_final_reward']:.2f} |
| RL Training Signal | reward_std={training_summary['reward_std']:.2f} | unsafe_before={training_summary['unsafe_rate_before']:.2f} | reward_end={training_summary['reward_end']:.2f} |
'''
display(Markdown(metrics_md))


## Before vs After Behavior

We replay one saved baseline trajectory and one saved compact-policy trajectory against the live environment so judges can see step-by-step actions and rewards side by side.


In [ ]:
import httpx

def replay_episode(task_id, seed, trajectory):
    client = httpx.Client(base_url='http://localhost:7860', timeout=20.0)
    reset = client.post('/reset', json={'task_id': task_id, 'seed': seed}).json()
    session_id = reset['session_id']
    records = []
    for idx, action in enumerate(trajectory, start=1):
        step = client.post('/step', json={'session_id': session_id, 'action': action}).json()
        records.append({
            'step': idx,
            'action_type': action['action_type'],
            'criterion_id': action.get('criterion_id'),
            'reward': step['reward']['total_reward'],
            'done': step['done'],
        })
        if step['done']:
            break
    return records

baseline_episode = base_eval['episodes'][0]
compact_episode = compact_eval['episodes'][0]

baseline_replay = replay_episode(baseline_episode['task_id'], baseline_episode['seed'], baseline_episode['trajectory'])
compact_replay = replay_episode(compact_episode['task_id'], compact_episode['seed'], compact_episode['trajectory'])

print('Baseline trajectory replay:')
print(json.dumps(baseline_replay, indent=2))
print('\nCompact policy trajectory replay:')
print(json.dumps(compact_replay, indent=2))


## Conclusion

- The baseline language model fails to complete the clinical workflow reliably.
- A real RL learning signal exists because reward variance is non-zero (`reward_std > 0`).
- The compact policy succeeds and demonstrates that the environment is learnable.
- The LLM training attempt was informative, but not fully solved.
